In [1]:
import numpy as np
from scipy.signal import butter, filtfilt, find_peaks


# ----------------------------
# Utility: tiers and framing
# ----------------------------

def duration_tier_scale(t, tier_points):
    """
    tier_points: list of (time_seconds, scale_factor)
    scale_factor > 1 => slower (longer), < 1 => faster (shorter)
    """
    tp = np.array([p[0] for p in tier_points], float)
    sp = np.array([p[1] for p in tier_points], float)
    return np.interp(t, tp, sp, left=sp[0], right=sp[-1])


def _frame_signal(x, frame_len, hop):
    n = len(x)
    n_frames = 1 + max(0, (n - frame_len) // hop)
    return np.stack([x[i * hop:i * hop + frame_len] for i in range(n_frames)], axis=0)


def _rms(frame):
    return np.sqrt(np.mean(frame * frame) + 1e-12)


def _bandpass(x, sr, lo=50.0, hi=900.0, order=3):
    nyq = 0.5 * sr
    b, a = butter(order, [lo / nyq, hi / nyq], btype="band")
    return filtfilt(b, a, x)


# ----------------------------
# F0 + voicing (simple ACF)
# ----------------------------

def _acf_pitch(frame, sr, fmin=60.0, fmax=400.0):
    frame = frame - np.mean(frame)
    if np.max(np.abs(frame)) < 1e-6:
        return 0.0, 0.0

    acf = np.correlate(frame, frame, mode="full")[len(frame) - 1:]
    acf[0] = 0.0
    m = np.max(acf)
    if m <= 0:
        return 0.0, 0.0
    acf = acf / (m + 1e-12)

    lag_min = int(sr / fmax)
    lag_max = int(sr / fmin)
    lag_max = min(lag_max, len(acf) - 1)
    if lag_min < 1 or lag_min >= lag_max:
        return 0.0, 0.0

    seg = acf[lag_min:lag_max]
    k = int(np.argmax(seg))
    lag = lag_min + k
    strength = float(seg[k])
    f0 = float(sr / lag) if strength > 0 else 0.0
    return f0, strength


def estimate_tracks(x, sr, frame_ms=30, hop_ms=10, fmin=60, fmax=400,
                    voiced_thresh=0.35):
    frame_len = int(sr * frame_ms / 1000)
    hop = int(sr * hop_ms / 1000)
    frames = _frame_signal(x, frame_len, hop)

    f0 = np.zeros(len(frames), float)
    strength = np.zeros(len(frames), float)
    rms = np.zeros(len(frames), float)

    for i, fr in enumerate(frames):
        rms[i] = _rms(fr)
        f0_i, s_i = _acf_pitch(fr, sr, fmin=fmin, fmax=fmax)
        f0[i] = f0_i
        strength[i] = s_i

    voiced = (strength >= voiced_thresh) & (f0 > 0)
    times = (np.arange(len(frames)) * hop + frame_len / 2) / sr
    return times, f0, strength, voiced, rms, frame_len, hop


# ----------------------------
# Region segmentation (voiced / unvoiced / silence)
# ----------------------------

def segment_regions(times, voiced, rms, silence_db=-40.0, min_region_ms=30):
    """
    Classify frames:
      - silence: rms below threshold (relative)
      - voiced: voiced==True
      - unvoiced: not voiced and not silence
    Then merge into contiguous regions.
    """
    # relative RMS threshold
    rms_db = 20 * np.log10(rms + 1e-12)
    mx = np.max(rms_db)
    silence = rms_db < (mx + silence_db)  # e.g., -40 dB below max

    cls = np.full(len(times), "unvoiced", dtype=object)
    cls[silence] = "silence"
    cls[voiced & (~silence)] = "voiced"

    # merge contiguous
    regions = []
    start = 0
    for i in range(1, len(cls)):
        if cls[i] != cls[i - 1]:
            regions.append((start, i - 1, cls[i - 1]))
            start = i
    regions.append((start, len(cls) - 1, cls[-1]))

    # merge too-short regions into neighbors (reduces chatter at boundaries)
    min_frames = max(1, int((min_region_ms / 1000) / (times[1] - times[0] + 1e-12)))
    merged = []
    for r in regions:
        if not merged:
            merged.append(r)
            continue
        s, e, c = r
        ps, pe, pc = merged[-1]
        if (e - s + 1) < min_frames:
            # merge into previous
            merged[-1] = (ps, e, pc)
        else:
            merged.append(r)

    return merged, cls


# ----------------------------
# Pulse detection (for voiced PSOLA)
# ----------------------------

def detect_pulses(x, sr, f0_times, f0_track, voiced_track, search_ms=6):
    xf = _bandpass(x, sr, 50, 900)
    n = len(x)
    t = np.arange(n) / sr
    f0_s = np.interp(t, f0_times, f0_track, left=0.0, right=0.0)
    v_s = np.interp(t, f0_times, voiced_track.astype(float), left=0.0, right=0.0) > 0.5

    pulses = []
    i = 0
    search = int(sr * search_ms / 1000)

    while i < n and not v_s[i]:
        i += 1

    while i < n:
        if not v_s[i] or f0_s[i] <= 0:
            i += 1
            continue

        period = int(sr / f0_s[i])
        if period < 16:
            i += 1
            continue

        lo = max(0, i - search)
        hi = min(n, i + search + 1)
        seg = xf[lo:hi]
        if len(seg) < 3:
            i += period
            continue

        peaks, _ = find_peaks(np.abs(seg))
        if len(peaks) == 0:
            pk = int(np.argmax(np.abs(seg)))
        else:
            center = i - lo
            pk = int(peaks[np.argmin(np.abs(peaks - center))])
        p = lo + pk
        pulses.append(p)
        i = p + period

    return np.array(sorted(set(pulses)), dtype=int)


# ----------------------------
# WSOLA for unvoiced / general pitch-preserving time-scaling
# ----------------------------

def wsola_time_scale(x, sr, scale, win_ms=40.0, hop_ms=10.0, search_ms=15.0):
    """
    WSOLA-like: choose best overlap position by maximizing correlation in a search region.
    scale < 1 => faster (shorter), > 1 => slower (longer)
    """
    if scale <= 0:
        raise ValueError("scale must be > 0")

    win = int(sr * win_ms / 1000)
    hop_out = int(sr * hop_ms / 1000)
    search = int(sr * search_ms / 1000)

    win = max(win, 128)
    hop_out = max(hop_out, 32)
    search = max(search, 32)

    w = np.hanning(win).astype(float)
    n = len(x)

    # input hop is scaled
    hop_in = int(round(hop_out / scale))
    hop_in = max(hop_in, 1)

    # output length estimate
    out_len = int(round(n * scale)) + win + 2
    y = np.zeros(out_len, float)

    # seed first frame
    in_pos = 0
    out_pos = 0
    y[out_pos:out_pos + win] += x[in_pos:in_pos + win] * w
    prev = y[out_pos:out_pos + win].copy()

    out_pos += hop_out
    in_pos += hop_in

    while (in_pos + win) < n and (out_pos + win) < len(y):
        # overlap region in output
        overlap_len = win - hop_out
        if overlap_len <= 32:
            overlap_len = win // 2

        # reference (tail of previous output window)
        ref = prev[hop_out:hop_out + overlap_len]
        ref = ref - np.mean(ref)

        # search around nominal in_pos
        best_k = 0
        best_score = -1e18

        lo = max(0, in_pos - search)
        hi = min(n - win - 1, in_pos + search)
        for cand in range(lo, hi + 1):
            seg = x[cand:cand + win]
            tail = (seg[:overlap_len] - np.mean(seg[:overlap_len]))
            denom = (np.linalg.norm(ref) * np.linalg.norm(tail) + 1e-12)
            score = float(np.dot(ref, tail) / denom)
            if score > best_score:
                best_score = score
                best_k = cand

        seg = x[best_k:best_k + win] * w

        # overlap-add at out_pos
        y[out_pos:out_pos + win] += seg
        prev = y[out_pos:out_pos + win].copy()

        out_pos += hop_out
        in_pos += hop_in

    # trim and normalize gently
    y = y[:max(out_pos, 1)]
    mx = np.max(np.abs(y)) + 1e-12
    y = y / mx * (np.max(np.abs(x)) + 1e-12)
    return y


# ----------------------------
# TD-PSOLA for voiced regions (global factor per region)
# ----------------------------

def td_psola_time_scale_segment(x, sr, pulses, scale, win_periods=2.0):
    """
    Time-scale a voiced segment by PSOLA using existing pitch marks.
    scale < 1 => faster, > 1 => slower
    """
    if len(pulses) < 3:
        # fallback: WSOLA (better than nothing)
        return wsola_time_scale(x, sr, scale)

    # local period from spacing
    dp = np.diff(pulses)
    dp = np.clip(dp, 20, int(sr / 50))  # guard

    # new pulse positions: spacing scaled by factor
    new_pulses = [int(pulses[0])]
    for k in range(1, len(pulses)):
        T = int(dp[k - 1])
        new_pulses.append(int(round(new_pulses[-1] + T * scale)))
    new_pulses = np.array(new_pulses, int)

    out_len = int(new_pulses[-1] + win_periods * np.max(dp) + 2)
    y = np.zeros(out_len, float)

    for k, p in enumerate(pulses):
        if k == 0:
            T = dp[0]
        elif k >= len(dp):
            T = dp[-1]
        else:
            T = dp[k - 1]
        half = int(win_periods * T / 2)
        half = max(half, 32)

        a = max(0, p - half)
        b = min(len(x), p + half)
        seg = x[a:b].copy()

        w = np.hanning(len(seg))
        seg *= w

        q = new_pulses[k]
        ya = max(0, q - (p - a))
        yb = ya + len(seg)
        if yb > len(y):
            y = np.pad(y, (0, yb - len(y)))
        y[ya:yb] += seg

    # normalize
    mx = np.max(np.abs(y)) + 1e-12
    y = y / mx * (np.max(np.abs(x)) + 1e-12)
    return y


# ----------------------------
# Glue: crossfade concatenation
# ----------------------------

def _crossfade_concat(chunks, sr, fade_ms=10.0):
    if not chunks:
        return np.array([], float)
    fade = int(sr * fade_ms / 1000)
    fade = max(fade, 32)

    y = chunks[0].astype(float)
    for nxt in chunks[1:]:
        nxt = nxt.astype(float)
        if len(y) < fade or len(nxt) < fade:
            y = np.concatenate([y, nxt])
            continue

        a = y[-fade:].copy()
        b = nxt[:fade].copy()
        w = np.linspace(0.0, 1.0, fade)
        y[-fade:] = a * (1 - w) + b * w
        y = np.concatenate([y, nxt[fade:]])
    return y


# ----------------------------
# Main: Praat-like rate change using regions
# ----------------------------

def praat_like_rate_change(x, sr, tier_points,
                           fmin=60, fmax=400,
                           voiced_thresh=0.35,
                           silence_db=-40.0,
                           fade_ms=12.0):
    """
    Hybrid approach:
      - segment into voiced/unvoiced/silence on frames
      - apply per-region mean scale from tier
      - voiced: PSOLA using pulses within region
      - unvoiced: WSOLA (keeps [s] continuous)
      - silence: WSOLA also works fine; or simple scaling
      - crossfade regions
    """
    # Analysis tracks
    t, f0, strength, voiced, rms, frame_len, hop = estimate_tracks(
        x, sr, fmin=fmin, fmax=fmax, voiced_thresh=voiced_thresh
    )

    regions, cls = segment_regions(t, voiced, rms, silence_db=silence_db, min_region_ms=30)

    # Detect pulses on full signal (we'll slice per region)
    pulses_all = detect_pulses(x, sr, t, f0, voiced)

    chunks = []
    for s_idx, e_idx, label in regions:
        # region time bounds in samples
        t0 = max(0.0, t[s_idx] - (frame_len / 2) / sr)
        t1 = min(len(x) / sr, t[e_idx] + (frame_len / 2) / sr)
        i0 = int(round(t0 * sr))
        i1 = int(round(t1 * sr))
        if i1 <= i0 + 64:
            continue

        seg = x[i0:i1]

        # mean scale in this region from tier
        mid_times = np.linspace(t0, t1, num=5)
        sc = float(np.mean(duration_tier_scale(mid_times, tier_points)))
        sc = max(0.5, min(2.0, sc))  # safety clamp

        if label == "voiced":
            # pulses that fall inside this segment
            p = pulses_all[(pulses_all >= i0) & (pulses_all < i1)] - i0
            yseg = td_psola_time_scale_segment(seg, sr, p, sc)
        else:
            # unvoiced + silence: WSOLA keeps fricatives continuous and pauses scaled
            yseg = wsola_time_scale(seg, sr, sc)

        chunks.append(yseg)

    y = _crossfade_concat(chunks, sr, fade_ms=fade_ms)
    return y


# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    import soundfile as sf  # pip install soundfile

    x, sr = sf.read("input.wav")
    if x.ndim > 1:
        x = x.mean(axis=1)

    dur = len(x) / sr

    # Example: speed up globally by 15% => scale=0.85 everywhere
    tier = [(0.0, 0.85), (dur, 0.85)]

    y = praat_like_rate_change(
        x, sr, tier,
        fmin=60, fmax=400,
        voiced_thresh=0.33,   # slightly more permissive helps
        silence_db=-45.0,     # treat quieter as silence
        fade_ms=15.0
    )

    sf.write("output_hybrid.wav", y, sr)
